# Establishing the baseline data and model for Monitoring

In [1]:
import json
import os
import sys
from pickle import dump

rootpath = os.path.dirname(os.getcwd())
sys.path.append(os.path.join(rootpath, "scripts"))
sys.path.append(os.path.join(rootpath, "src"))

In [2]:
from gcp_functions import load_pickle_from_gcs, read_file_as_df, upload_directory
from load_configs import Configs

from data import (
    build_features,
    create_X_y_multistep,
)

configs = Configs()

loading gcs configs for None


In [3]:
data_ref_path = {
    "bucket": configs.cloud["gcs"]["data_monitoring_bucket"],
    "prefix": "cleaned_samples_prod",
    "fname": "Kaggle_Access_2025-07-22_WSPall_from_2020-07-22.parquet",
}

model_ref_path = {
    "bucket": "stocks-forecasting-models-dev",
    "prefix": "extracted_model",
    "fname": "model.pkl",
}

In [4]:
df = read_file_as_df(
    configs.cloud["gcs"]["project"],
    data_ref_path["bucket"],
    data_ref_path["prefix"] + "/" + data_ref_path["fname"],
)

In [5]:
estimator = load_pickle_from_gcs(
    configs.cloud["gcs"]["project"],
    model_ref_path["bucket"],
    model_ref_path["prefix"] + "/" + model_ref_path["fname"],
)

In [6]:
params = {"CldrFeats": "True", "ModReg": "True", "lags": "3", "steps": "5"}
TARGET = "returns"

In [7]:
df_feats, _features2scale = build_features(
    df, lags=int(params["lags"]), split="train", CldrFeats=params["CldrFeats"]
)
X, y = create_X_y_multistep(df_feats, steps=int(params["steps"]), target=TARGET)

In [8]:
# Evidently cannot deal with multi-output yet, so we need to merge the first step y_test
# merge X_test and first step y_test, and into one dataframe
y_hat = estimator.predict(X)
ref_data = X.copy()
ref_data["target"] = y["y_step_1"]
ref_data["prediction"] = y_hat[:, 0]
ref_data.head()

returns_lag1  returns_lag2  returns_lag3  \
Ticker Date                                                            
AAPL   2020-10-07 04:00:00     -0.016684      0.029516     -0.029871   
       2020-10-08 04:00:00      0.000957     -0.016684      0.029516   
       2020-10-09 04:00:00     -0.017098      0.000957     -0.016684   
       2020-10-12 04:00:00     -0.059727     -0.017098      0.000957   
       2020-10-13 04:00:00      0.027250     -0.059727     -0.017098   

                                SMA_10      SMA_50      EMA_12      EMA_26  \
Ticker Date                                                                  
AAPL   2020-10-07 04:00:00  111.967146  112.990314  112.074643  112.172478   
       2020-10-08 04:00:00  112.630162  113.385197  112.206046  112.228499   
       2020-10-09 04:00:00  113.090836  113.796820  112.619464  112.425889   
       2020-10-12 04:00:00  114.018076  114.156935  114.092060  113.149256   
       2020-10-13 04:00:00  114.706629  114.399718  114.839425  113.578935   

                                MACD  MACD_Signal  MACD_Hist  ...    BB_Lower  \
Ticker Date                                                   ...               
AAPL   2020-10-07 04:00:00 -0.097835    -0.376234   0.278399  ...  104.864233   
       2020-10-08 04:00:00 -0.022453    -0.305477   0.283025  ...  104.862460   
       2020-10-09 04:00:00  0.193576    -0.205667   0.399242  ...  104.847776   
       2020-10-12 04:00:00  0.942805     0.024027   0.918777  ...  103.574398   
       2020-10-13 04:00:00  1.260490     0.271320   0.989170  ...  103.206383   

                            BB_Width     RSI_14    ROC_10  sin(1,freq=ME)  \
Ticker Date                                                                 
AAPL   2020-10-07 04:00:00  0.105939  56.391611  0.074309        0.937752   
       2020-10-08 04:00:00  0.107213  62.065879  0.062373        0.988468   
       2020-10-09 04:00:00  0.111638  60.616333  0.041771        0.998717   
       2020-10-12 04:00:00  0.142001  66.500645  0.082116        0.790776   
       2020-10-13 04:00:00  0.153126  69.015202  0.061443        0.651372   

                            cos(1,freq=ME)  sin(1,freq=W-SUN)  \
Ticker Date                                                     
AAPL   2020-10-07 04:00:00        0.347305           0.974928   
       2020-10-08 04:00:00        0.151428           0.433884   
       2020-10-09 04:00:00       -0.050649          -0.433884   
       2020-10-12 04:00:00       -0.612106           0.000000   
       2020-10-13 04:00:00       -0.758758           0.781831   

                            cos(1,freq=W-SUN)    target  prediction  
Ticker Date                                                          
AAPL   2020-10-07 04:00:00          -0.222521  0.000957    0.001628  
       2020-10-08 04:00:00          -0.900969 -0.017098   -0.014491  
       2020-10-09 04:00:00          -0.900969 -0.059727   -0.052609  
       2020-10-12 04:00:00           1.000000  0.027250    0.024200  
       2020-10-13 04:00:00           0.623490 -0.000743   -0.002195  

[5 rows x 23 columns]

In [9]:
ref_path = os.path.join(rootpath, "ref_data_model")

os.makedirs(ref_path, exist_ok=True)

# save params as json
with open(os.path.join(ref_path, "params.json"), "w") as f:
    json.dump(params, f)
# save model pickle bin
with open(os.path.join(ref_path, "model.pkl"), "wb") as f:
    dump(estimator, f)
# save data as parquet
ref_data.to_parquet(os.path.join(ref_path, "data.parquet"))

In [10]:
upload_directory(
    project_id=configs.cloud["gcs"]["project"],
    bucket_name=configs.cloud["gcs"]["data_monitoring_bucket"],
    folder="ref_data_model",
    local_dir=ref_path,
)

Uploaded /home/onur/WORK/DS/repos/stocks_forecasting_live/ref_data_model/data.parquet to gs://stocks-forecasting-data-monitoring/ref_data_model/data.parquet
Uploaded /home/onur/WORK/DS/repos/stocks_forecasting_live/ref_data_model/params.json to gs://stocks-forecasting-data-monitoring/ref_data_model/params.json
Uploaded /home/onur/WORK/DS/repos/stocks_forecasting_live/ref_data_model/model.pkl to gs://stocks-forecasting-data-monitoring/ref_data_model/model.pkl
Successfully uploaded 3 file(s) to gs://stocks-forecasting-data-monitoring/ref_data_model
